# 02 — M3/M4: V-JEPA ve V-JEPA2

Bu notebook görüntü patch'lerinden uzay-zamansal tubelet'lere geçer. Önce küçük ve okunabilir V-JEPA ile tensor akışını doğrular, sonra resmî V-JEPA/V-JEPA2 feature extraction hücrelerine geçer.

Ana soru: bir video representation modeli zamansal sıraya duyarlı olabilir; fakat bu onu neden otomatik olarak action-conditioned, causal rollout world model yapmaz?

In [ ]:
from pathlib import Path
import os, sys
override = os.environ.get('JEPA_LAB_ROOT')
candidates = ([Path(override)] if override else []) + [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/jepa-study'), Path('/kaggle/working/I-JEPA'), Path('/content/jepa-study'), Path('/content/I-JEPA')]
ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Repo bulunamadı; JEPA_LAB_ROOT değişkenini ayarlayın.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('repo:', ROOT)

In [ ]:
%matplotlib inline
from pprint import pprint
import subprocess
import torch
from torch.utils.data import DataLoader
from jepa_lab.device import seed_everything, select_device
from jepa_lab.foundations import video_token_grid, video_token_count
from jepa_lab.runlog import load_yaml

seed_everything(42)
device = select_device('auto')
config = load_yaml(ROOT / 'configs/vjepa_tiny.yaml')
print('device:', device)
pprint(config)

## Tahmin 1 — Tubelet token hesabı

`[B,T,C,H,W]=[1,16,3,224,224]` ve tubelet `2×16×16`. Conv3D çıktı grid'ini ve toplam token sayısını çalıştırmadan hesaplayın. Temporal tubelet'in iki komşu frame'i tek token'a toplamasının olası avantaj/dezavantajı nedir?

In [ ]:
grid = video_token_grid(frames=16, image_size=224, tubelet_size=2, patch_size=16)
tokens = video_token_count(frames=16, image_size=224, tubelet_size=2, patch_size=16)
print('grid:', grid, 'tokens:', tokens)
assert grid == (8, 14, 14) and tokens == 1568

## Sentetik hareket ve full-time tube mask

### Tahmin 2

V-JEPA'nın short/long spatial blokları neden temporal eksenin tamamına uzatılır? Sekiz küçük + iki büyük maskenin birleşiminin yaklaşık `%90` masking vermesini bekliyoruz. Her temporal slice aynı mı görünmeli?

In [ ]:
import matplotlib.pyplot as plt
from jepa_lab.datasets import MovingShapesDataset
from jepa_lab.video_jepa import TinyVideoJEPA, generate_future_mask, generate_tube_masks
from jepa_lab.visualization import plot_tube_mask

sample = MovingShapesDataset(samples=8, frames=8, image_size=112, seed=42)[0]
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for t, axis in enumerate(axes):
    axis.imshow(sample['frames'][t].permute(1, 2, 0))
    axis.set_title(f't={t}')
    axis.axis('off')
plt.tight_layout()

mask = generate_tube_masks((4, 7, 7), seed=42)
display(plot_tube_mask(mask.target))
print('coverage:', mask.coverage)
assert all(torch.equal(mask.target[0], mask.target[t]) for t in range(1, 4))

### İncele 2

Full-time tube, aynı spatial bölgeyi zaman boyunca gizler; predictor yalnız komşu frame'deki aynı piksel konumunu kopyalayamaz. Küçük grid'de on mask tüm alanı kapatabileceği için laboratuvar implementasyonu en az `%10` context kalacak şekilde union'ı deterministik sınırlar.

## V-JEPA forward: `[B,T,C,H,W] → [B,N,D]`

### Tahmin 3

Teacher tam videoyu mu, yalnız target tubelet'leri mi encoder'a verir? Target seçiminin encoder'dan önce yapılması hangi bağlamsal bilgiyi yok ederdi?

In [ ]:
video_model = TinyVideoJEPA(
    num_frames=8, image_size=112, tubelet_size=(2, 16, 16),
    embed_dim=48, encoder_depth=1, encoder_heads=3,
    predictor_dim=48, predictor_depth=1, predictor_heads=3,
).to(device)
frames = sample['frames'].unsqueeze(0).to(device)
with torch.no_grad():
    full_teacher = video_model.encode(frames)
output = video_model(frames, mask.target.to(device))
print('input canonical:', tuple(frames.shape))
print('latent grid:', video_model.grid_size)
print('full teacher:', tuple(full_teacher.shape))
print('prediction:', tuple(output.prediction.shape), 'target:', tuple(output.target.shape))
print('L1:', float(output.loss.detach()))
assert full_teacher.shape == (1, 196, 48)
assert output.prediction.shape == output.target.shape

In [ ]:
video_opt = torch.optim.AdamW([*video_model.context_encoder.parameters(), *video_model.predictor.parameters()], lr=1e-3)
video_opt.zero_grad(set_to_none=True)
output = video_model(frames, mask.target.to(device))
output.loss.backward()
target_has_grad = any(p.grad is not None for p in video_model.target_encoder.parameters())
student_grad = sum(float(p.grad.square().sum()) for p in video_model.context_encoder.parameters() if p.grad is not None) ** 0.5
print({'target_has_grad': target_has_grad, 'student_grad_norm': student_grad})
assert not target_has_grad and student_grad > 0

## Hızlı sentetik V-JEPA eğitimi

`configs/vjepa_tiny.yaml` uzun koşu için `8×112×112`, width `192`, 5000 adım tanımlar. Bu hücre tensor mekanizmasını hızlı doğrulamak için `8×64×64`, width `48`, 20 adım kullanır. Sonuç bir benchmark değildir.

In [ ]:
from jepa_lab.experiments import train_video_steps

quick_dataset = MovingShapesDataset(samples=128, frames=8, image_size=64, seed=42)
quick_loader = DataLoader(quick_dataset, batch_size=4, shuffle=False, num_workers=0)
trained_video = TinyVideoJEPA(
    num_frames=8, image_size=64, tubelet_size=(2, 16, 16),
    embed_dim=48, encoder_depth=1, encoder_heads=3,
    predictor_dim=48, predictor_depth=1, predictor_heads=3,
)
quick_opt = torch.optim.AdamW([*trained_video.context_encoder.parameters(), *trained_video.predictor.parameters()], lr=1e-3)
video_metrics = train_video_steps(trained_video, quick_loader, quick_opt, steps=20, device=device, seed=42)
pprint(video_metrics.as_dict())

# Aynı target latent'lerini batch içinde kaydırmak, doğru clip-target eşleşmesini bozar.
comparison_batch = next(iter(quick_loader))['frames'].to(device)
comparison_mask = generate_tube_masks(trained_video.grid_size, seed=404).target.to(device)
with torch.inference_mode():
    comparison = trained_video(comparison_batch, comparison_mask)
    matched_l1 = torch.nn.functional.l1_loss(comparison.prediction, comparison.target)
    shuffled_l1 = torch.nn.functional.l1_loss(
        comparison.prediction, torch.roll(comparison.target, shifts=1, dims=0)
    )
print({
    'matched_target_L1': float(matched_l1), 'shuffled_target_L1': float(shuffled_l1),
    'matched_is_lower': bool(matched_l1 < shuffled_l1),
})

RUN_5000_STEP_VJEPA = False
full_video = None
if RUN_5000_STEP_VJEPA:
    from jepa_lab.checkpointing import save_training_checkpoint
    from jepa_lab.runlog import build_run_summary, write_run_summary
    video_data = MovingShapesDataset(samples=4096, frames=8, image_size=112, seed=42)
    video_loader = DataLoader(video_data, batch_size=8, shuffle=True, num_workers=2)
    vm = config['model']
    full_video = TinyVideoJEPA(
        num_frames=8, image_size=112, tubelet_size=(2, 16, 16),
        embed_dim=vm['embed_dim'], encoder_depth=vm['encoder_depth'],
        encoder_heads=vm['encoder_heads'], predictor_dim=vm['predictor_dim'],
        predictor_depth=vm['predictor_depth'], predictor_heads=vm['predictor_heads'],
    )
    full_opt = torch.optim.AdamW(
        [*full_video.context_encoder.parameters(), *full_video.predictor.parameters()],
        lr=config['optimization']['learning_rate'],
        weight_decay=config['optimization']['weight_decay'],
    )
    full_metrics = train_video_steps(
        full_video, video_loader, full_opt, steps=5000, device=device, seed=42,
        ema_start=0.998, ema_end=1.0,
    )
    checkpoint = save_training_checkpoint(
        ROOT / 'runs/vjepa_tiny_5000.pt', full_video, full_opt, step=5000,
        metadata={'seed': 42, 'dataset': 'moving-shapes', 'scope': 'teaching-scale'},
    )
    summary = build_run_summary(config, full_metrics.as_dict(), {'checkpoint': str(checkpoint)})
    print('summary:', write_run_summary(summary, ROOT / 'runs'))
    pprint(full_metrics.as_dict())
else:
    print('5000-step koşu atlandı; Kaggle GPU hazır olduğunda bayrağı açın.')

# Acceptance kıyası varsa 5.000-step encoder'ı, yoksa yalnız quick modeli kullanır.
acceptance_video = full_video if full_video is not None else trained_video
acceptance_size = 112 if full_video is not None else 64
acceptance_loader = DataLoader(
    MovingShapesDataset(samples=32, frames=8, image_size=acceptance_size, seed=404),
    batch_size=16, shuffle=False, num_workers=0,
)
acceptance_frames = next(iter(acceptance_loader))['frames'].to(device)
acceptance_mask = generate_tube_masks(acceptance_video.grid_size, seed=404).target.to(device)
with torch.inference_mode():
    acceptance_output = acceptance_video(acceptance_frames, acceptance_mask)
    acceptance_matched = torch.nn.functional.l1_loss(
        acceptance_output.prediction, acceptance_output.target
    )
    acceptance_shuffled = torch.nn.functional.l1_loss(
        acceptance_output.prediction, torch.roll(acceptance_output.target, shifts=1, dims=0)
    )
print({
    'model_budget': 5000 if full_video is not None else 20,
    'matched_target_L1': float(acceptance_matched),
    'shuffled_target_L1': float(acceptance_shuffled),
    'passes_matched_lower': bool(acceptance_matched < acceptance_shuffled),
})

## Temporal perturbation

### Tahmin 4

Aynı klibin normal, ters, shuffled ve tek-frame-tekrar sürümlerinin mean-pooled feature cosine'ları nasıl sıralanır? Random-init veya 20-step modelde bu sıralamanın anlamsal kanıt sayılamayacağını da açıklayın.

In [ ]:
from jepa_lab.adapters import temporal_variants, cosine_to_reference

probe_frames = MovingShapesDataset(samples=1, frames=8, image_size=64, seed=7)[0]['frames'].unsqueeze(0).to(device)
variants = temporal_variants(probe_frames)
with torch.inference_mode():
    variant_features = {name: trained_video.encode(value).cpu() for name, value in variants.items()}
temporal_cosines = cosine_to_reference(variant_features)
pprint(temporal_cosines)

## Non-causal multiblock ↔ causal future ablation

Aşağıdaki causal maske yalnız grid'in ikinci temporal yarısını target yapar. Tek forward loss kıyaslaması eğitim sonucu değildir; gerçek ablation için iki modeli aynı seed/step bütçesiyle ayrı eğitin. Non-causal V-JEPA representation öğrenirken geçmiş ve gelecek bağlamı kullanabilir; robot planning rollout'u ise gelecek gözlemi göremez.

In [ ]:
causal_future = generate_future_mask(trained_video.grid_size)
noncausal = generate_tube_masks(trained_video.grid_size, seed=123).target
with torch.inference_mode():
    causal_loss = float(trained_video(probe_frames, causal_future.to(device)).loss)
    noncausal_loss = float(trained_video(probe_frames, noncausal.to(device)).loss)
print({'same_model_causal_future_L1': causal_loss, 'same_model_noncausal_L1': noncausal_loss})

RUN_CAUSAL_ABLATION = False
CAUSAL_ABLATION_STEPS = 100
if RUN_CAUSAL_ABLATION:
    def fit_mask_ablation(mask_factory):
        seed_everything(42)
        candidate = TinyVideoJEPA(
            num_frames=8, image_size=64, tubelet_size=(2, 16, 16),
            embed_dim=48, encoder_depth=1, encoder_heads=3,
            predictor_dim=48, predictor_depth=1, predictor_heads=3,
        )
        candidate_opt = torch.optim.AdamW(
            [*candidate.context_encoder.parameters(), *candidate.predictor.parameters()], lr=1e-3
        )
        return train_video_steps(
            candidate, quick_loader, candidate_opt, steps=CAUSAL_ABLATION_STEPS,
            device=device, seed=42, mask_factory=mask_factory,
        ).as_dict()
    noncausal_metrics = fit_mask_ablation(None)
    causal_metrics = fit_mask_ablation(lambda grid, _seed: generate_future_mask(grid))
    pprint({'noncausal_multiblock': noncausal_metrics, 'causal_future': causal_metrics})
else:
    print('Eşit bütçeli kısa eğitim için RUN_CAUSAL_ABLATION=True yapın.')

## Motion-direction probe — isteğe bağlı kısa değerlendirme

Encoder dondurulur; yalnız linear head eğitilir. Kabul ölçütü, aynı mimarideki random-init encoder'ın probe skorunu en az 10 yüzde puan aşmaktır. Dört sınıflı görevin `%25` şans düzeyi ayrıca raporlanabilir ama random-encoder kontrolünün yerini tutmaz. 20-step encoder bunu geçmeyebilir; o durumda 5000-step Kaggle koşusunun feature'larıyla tekrar deneyin.

In [ ]:
RUN_DIRECTION_PROBE = False
if RUN_DIRECTION_PROBE:
    from jepa_lab.evaluation import evaluate_representation
    probe_encoder = full_video if full_video is not None else trained_video
    probe_size = 112 if full_video is not None else 64
    @torch.inference_mode()
    def freeze_features(encoder, dataset):
        loader = DataLoader(dataset, batch_size=16, shuffle=False)
        xs, ys = [], []
        for batch in loader:
            xs.append(encoder.encode(batch['frames'].to(device)).mean(1).cpu())
            ys.append(batch['label'])
        return torch.cat(xs), torch.cat(ys)
    train_data = MovingShapesDataset(samples=256, frames=8, image_size=probe_size, seed=1000)
    val_data = MovingShapesDataset(samples=128, frames=8, image_size=probe_size, seed=9000)
    x_train, y_train = freeze_features(probe_encoder, train_data)
    x_val, y_val = freeze_features(probe_encoder, val_data)
    trained_scores = evaluate_representation(x_train, y_train, x_val, y_val, k=20, probe_steps=200)
    seed_everything(42)
    if full_video is None:
        random_video = TinyVideoJEPA(
            num_frames=8, image_size=64, tubelet_size=(2, 16, 16),
            embed_dim=48, encoder_depth=1, encoder_heads=3,
            predictor_dim=48, predictor_depth=1, predictor_heads=3,
        ).to(device)
    else:
        random_video = TinyVideoJEPA(
            num_frames=8, image_size=112, tubelet_size=(2, 16, 16),
            embed_dim=vm['embed_dim'], encoder_depth=vm['encoder_depth'],
            encoder_heads=vm['encoder_heads'], predictor_dim=vm['predictor_dim'],
            predictor_depth=vm['predictor_depth'], predictor_heads=vm['predictor_heads'],
        ).to(device)
    random_train_x, random_train_y = freeze_features(random_video, train_data)
    random_val_x, random_val_y = freeze_features(random_video, val_data)
    random_scores = evaluate_representation(
        random_train_x, random_train_y, random_val_x, random_val_y, k=20, probe_steps=200
    )
    probe_deltas = {
        'knn': trained_scores.knn_accuracy - random_scores.knn_accuracy,
        'linear': trained_scores.linear_accuracy - random_scores.linear_accuracy,
    }
    pprint({
        'trained': trained_scores.as_dict(), 'random_encoder': random_scores.as_dict(),
        'trained_minus_random': probe_deltas,
        'passes_knn_plus_10': probe_deltas['knn'] >= 0.10,
        'passes_linear_plus_10': probe_deltas['linear'] >= 0.10,
    })
else:
    print('Atlandı. Probe için RUN_DIRECTION_PROBE=True yapın.')

## Resmî V-JEPA v1 — ayrı process

İlk hücre checkpoint indirmeden pinned Meta kodunun shape smoke test'ini yapar. Pretrained ViT-L deneyi için `configs/checkpoints.yaml` adresindeki dosyayı `checkpoints/` altına indirin; sonra `vit_large`, 16 frame, 224 çözünürlükle çalıştırın. Büyük model hücreleri Kaggle GPU içindir.

In [ ]:
RUN_OFFICIAL_VJEPA_SHAPE = False
if RUN_OFFICIAL_VJEPA_SHAPE:
    subprocess.run([
        sys.executable, str(ROOT / 'scripts/official_vjepa_features.py'),
        '--device', str(device), '--model', 'vit_tiny', '--frames', '8', '--image-size', '112',
    ], cwd=ROOT, check=True)
else:
    print('Atlandı. RUN_OFFICIAL_VJEPA_SHAPE=True yapın.')

In [ ]:
RUN_PRETRAINED_VJEPA_VITL = False
vjepa_checkpoint = ROOT / 'checkpoints/vitl16.pth.tar'
if RUN_PRETRAINED_VJEPA_VITL:
    if not vjepa_checkpoint.is_file():
        raise FileNotFoundError("Önce resmî V-JEPA checkpoint'ini checkpoints/ altına indirin.")
    subprocess.run([
        sys.executable, str(ROOT / 'scripts/official_vjepa_features.py'),
        '--device', str(device), '--model', 'vit_large', '--frames', '16', '--image-size', '224',
        '--checkpoint', str(vjepa_checkpoint),
        '--output', str(ROOT / 'runs/vjepa_v1_temporal_features.npz'),
    ], cwd=ROOT, check=True)
else:
    print('Atlandı. Kaggle GPU + checkpoint hazır olduğunda bayrağı açın.')

## V-JEPA2: 3D RoPE ve resmî Hugging Face encoder

V-JEPA2'de V-JEPA v1'e göre izleyeceğimiz üç değişiklik: absolute 3D sin-cos yerine **3D RoPE**, daha büyük image/video karışımı ve progressive temporal/spatial resolution. Action-free encoder pretraining ile action-conditioned post-training'i birbirine karıştırmayın.

Aşağıdaki hücre `facebook/vjepa2-vitl-fpc64-256` ağırlıklarını indirir ve büyük VRAM/RAM kullanır. Önce `pip install -e '.[hf]'`; Kaggle GPU'da çalıştırın. Feature'lar `.npz` ile process sınırının dışına çıkar.

In [ ]:
RUN_OFFICIAL_VJEPA2 = False
if RUN_OFFICIAL_VJEPA2:
    subprocess.run([
        sys.executable, str(ROOT / 'scripts/official_vjepa2_features.py'),
        '--device', str(device), '--frames', '64', '--image-size', '256',
        '--output', str(ROOT / 'runs/vjepa2_temporal_features.npz'),
    ], cwd=ROOT, check=True)
else:
    print('Atlandı. Hugging Face extra + yeterli GPU hazır olduğunda bayrağı açın.')

In [ ]:
from jepa_lab.evaluation import temporal_perturbation_summary

v1_archive = ROOT / 'runs/vjepa_v1_temporal_features.npz'
v2_archive = ROOT / 'runs/vjepa2_temporal_features.npz'
if v1_archive.is_file() and v2_archive.is_file():
    pprint({'V-JEPA-v1': temporal_perturbation_summary(v1_archive).as_dict()})
    pprint({'V-JEPA2': temporal_perturbation_summary(v2_archive).as_dict()})
else:
    print('Karşılaştırma için iki resmî feature export dosyasını da üretin.')

## M3/M4 geçiş kontrolü

- `[B,T,C,H,W] → Conv3D tubelet → [B,N,D]` akışını ve 1568 token hesabını açıklayın.
- Teacher tam klibi görür; target seçimi latent çıktıdadır.
- Normal/reversed/shuffled/static sonuçlarında random-init ile pretrained kanıtı ayırın.
- V-JEPA v1 güçlü non-causal representation modelidir; action-conditioned causal dynamics değildir.
- V-JEPA2 action-free encoder pretraining ile V-JEPA2-AC post-training iki ayrı aşamadır.